# Evaluate Phase 2-beta v1 (Qwen-0.5B full FT)Evaluates the v1 baseline on the LIARArg parse task. Hits about 62% empty rate, which is why we did not run v1 through the full Phase 1 integration. This produced the v1 = 0.108 number in the paper's ablation table.Recommended to rename this file to `v1_indomain_eval.ipynb`.

In [ ]:
%%bash
# Line count = rows finished (each row is fsync'd)
wc -l ~/argument-aware-rag/phase2_data_liar/parser_preds_phase2beta_qwen0.5b.jsonl 2>/dev/null
echo ""

# Is the process alive + how long has it been going?
# (Look for the python3 with PYTHONPATH/python3 -c "..." heredoc — not just "python3")
ps aux | grep -E "python3.*phase2_beta|python3 <<" | grep -v grep | head -3
echo ""

# GPU activity
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

In [ ]:
%%bash
JSONL=~/argument-aware-rag/phase2_data_liar/parser_preds_phase2beta_qwen0.5b.jsonl
echo "=== last 3 rows in $(basename $JSONL) ==="
tail -3 $JSONL | python3 -c "
import json, sys
for i, line in enumerate(sys.stdin, 1):
    rec = json.loads(line)
    p = rec['prediction']
    n_c = len(p['claim_components'])
    n_p = len(p['premise_components'])
    n_r = len(p['relations'])
    status = 'REAL' if (n_c + n_p) > 0 else 'EMPTY'
    print(f'--- entry {i}: row_id={rec[\"row_id\"]} [{status}] ---')
    print(f'  claims:    {n_c}')
    print(f'  premises:  {n_p}')
    print(f'  relations: {n_r}')
    if n_c > 0:
        print(f'  first claim: {p[\"claim_components\"][0][\"text\"][:120]}')
    if n_p > 0:
        print(f'  first premise: {p[\"premise_components\"][0][\"text\"][:120]}')
    print()
"
echo ""
echo "=== aggregate empty rate so far ==="
python3 -c "
import json
total, empty = 0, 0
with open('phase2_data_liar/parser_preds_phase2beta_qwen0.5b.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        total += 1
        p = rec['prediction']
        if (len(p['claim_components'])==0 and len(p['premise_components'])==0):
            empty += 1
print(f'  {total} rows done  |  empty: {empty} ({100*empty/total:.1f}%)  |  real: {total-empty}')
"

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 << 'PY'
import sys, json, time
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student
from src.phase2.dataset import read_jsonl
from src.phase2.evaluate import evaluate_corpus

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b.yaml')
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print(f"loaded: use_cache={student._model.config.use_cache}  max_target_len={cfg.student.max_target_len}")

MAX_PER_DOMAIN = 100   # cap for speed; remove for full eval later

print("\n=== Phase 2-β v1 parser-level eval (capped) ===\n")
results_summary = {}
for dom in ('abstrct', 'microtext', 'cdcp', 'perspectrum'):
    recs = read_jsonl(f'phase2_data/unified/{dom}_test.jsonl')
    if not recs:
        print(f"  {dom}: no test file, skipping")
        continue
    recs = recs[:MAX_PER_DOMAIN]
    t0 = time.time()
    preds = []
    n_empty = 0
    for i, r in enumerate(recs, 1):
        try:
            pred, _ = student.predict(r['input'])
            if len(pred['claim_components'])==0 and len(pred['premise_components'])==0:
                n_empty += 1
            preds.append(pred)
        except Exception as e:
            preds.append({'claim_components':[],'premise_components':[],
                          'citation_components':[],'relations':[]})
            n_empty += 1
            print(f"  {dom} row {i} ERROR: {e}")
        if i % 25 == 0:
            elapsed = time.time() - t0
            rate = i / max(elapsed, 1e-6)
            print(f"  {dom}  {i}/{len(recs)}  ({elapsed:.0f}s, {rate:.2f}/s)")
    metrics = evaluate_corpus(recs, preds, threshold=0.5)
    elapsed = time.time() - t0
    print(f"\n  ── {dom} ── n={len(recs)}  empty={n_empty} ({100*n_empty/len(recs):.0f}%)  "
          f"({elapsed:.0f}s)")
    print(f"    component macro-F1:  {metrics['macro_component_f1']:.3f}")
    print(f"    relation F1:         {metrics['relation_f1']['f1']:.3f}")
    print(f"    claim F1:            {metrics['component_f1']['claim']['f1']:.3f}")
    print(f"    premise F1:          {metrics['component_f1']['premise']['f1']:.3f}")
    print(f"    citation F1:         {metrics['component_f1']['citation']['f1']:.3f}")
    print()
    results_summary[dom] = {
        'n': len(recs), 'empty': n_empty,
        'component_f1': metrics['macro_component_f1'],
        'relation_f1': metrics['relation_f1']['f1'],
    }

print("=== SUMMARY ===")
for dom, m in results_summary.items():
    print(f"  {dom:15s}  comp-F1={m['component_f1']:.3f}  rel-F1={m['relation_f1']:.3f}  empty={m['empty']}/{m['n']}")
PY